# 04 - Trend analysis (Phase 4)

Pixel-wise **Mann-Kendall** and **Sen's slope** on the ANNUAL composite series,
Benjamini-Hochberg FDR correction, a decadal comparison, and
autocorrelation-robust trend tests.

## Why this notebook is the one that measures warming

Phase 3's UTFVI epoch maps **cannot** show warming. UTFVI's reference is the
year's own mean, so a uniformly warming city produces no class change at all;
epoch-to-epoch drift is *redistribution*. **Sen's slope here is the warming**,
and nothing in this notebook "confirms" the UTFVI maps.

## This notebook runs in TWO PARTS

| | What it does | Needs Earth Engine? |
|---|---|---|
| **Part 1** | builds the trend products and submits batch exports | yes |
| *(wait)* | you download the results from Drive into `data/interim/` | — |
| **Part 2** | every figure and statistic, computed locally in numpy | **no** |

That split is not stylistic. Colab runs 10-13 established that **no interactive
question about the trend graph is affordable** - not `reduceRegion`, not
`sampleRectangle`, not even `bandNames()`. A batch `Export` task has no such
ceiling. So Part 1 asks Earth Engine for nothing it does not have to, and Part 2
works from the downloaded rasters, which also makes better figures than
`getThumbURL` previews.

## Two corrections to the GEE community tutorial

This project follows the official *Non-Parametric Trend Analysis* tutorial for
the MK variance, Z and p - with two deliberate corrections, both implemented in
`src/colombo_uhi/trends.py`:

| # | Tutorial | Why it is wrong here | What we do |
|---|---|---|---|
| 1 | `sign = diff.clamp(-1,1).int()` | `.int()` truncates toward zero, so a **+0.3 degC** year-to-year difference becomes sign **0**. Most annual LST differences in Colombo are well under 1 degC, so this collapses most of S. The tutorial states it is for *"discrete data (i.e. not floating point)"*. | `diff.gt(0).subtract(diff.lt(0))` - exact for any float |
| 2 | `p = 1 - Phi(abs(z))`, thresholded at 0.025 | That is the **one-sided** p. Benjamini-Hochberg needs **two-sided** input; the one-sided form halves every p and roughly doubles the significant area. | `mk_p_two_sided = 2*(1 - Phi(abs(Z)))` |

A third: the tutorial's tie correction detects ties by **exact float equality**.
On continuous LST there are effectively zero ties, so the term is measurably
zero while the `arraySort` machinery is expensive. `trends.tie_correction` is
off, with the reasoning recorded in `config/params.yaml`.

## Non-negotiable caveats

1. This is **land surface temperature**, never air temperature.
2. Every trend product ships its per-pixel valid-**year** count (`n_years`).
3. `n_years` is never masked - an excluded pixel reads as *excluded*, not *missing*.
4. Landsat is a single ~10:30 overpass. Night trends come only from MODIS.

---
# PART 1 - build and submit

Everything here is cheap or batched. Nothing in Part 1 interrogates the trend
graph, and any step that turns out to be too expensive prints a note and
continues rather than aborting the run.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00-03 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "valid_obs_required", "single_overpass", "fdr_dependence"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 0 - prove the loaded code is current

Two checks, because the usual one is **not enough here**.

The name-based guard catches a stale `src/`. But some Phase 4 changes are
**additive** - `composites.annual_composites` gained a `series_basis` property
and renamed nothing - so a name check passes against older code and then fails
much later with a confusing message. The second cell proves the **setter** is
present and that the measured reducer names are in `params`.

This cell also defines `_try_ee`, the helper every remaining interactive Earth
Engine call goes through, and the cheap constants the later cells need. Both are
here so a skipped step can never cascade into a `NameError` - which is exactly
what ended the previous run.

In [ ]:
# COLAB: RUN THIS CELL
# Every import Parts 1 AND 2 need, in one place. Part 2 is meant to be run as a
# separate session, so it must not depend on an import that happens to live in a
# Part 1 cell.
import glob
import time

import ee
import numpy as np
import pandas as pd
from IPython.display import Image, display

from colombo_uhi import aoi, composites, exports, landcover, landsat, modis, trends, uhi_metrics, viz

# Fail fast and legibly if a STALE colombo_uhi is loaded (see the purge above).
#
# THE RULE, learned the hard way in run 8: every entry must name a function
# introduced by the MOST RECENT revision of that module. Listing only functions
# that also existed in the previous revision makes this guard VACUOUS -- it
# passes, and you get old figures from new notebook cells with no error anywhere.
_required = {
    "aoi": ["rural_reference", "static_water_mask", "lcz_scope_geometry"],
    "composites": ["annual_composites", "warn_if_counts_are_empty"],
    "uhi_metrics": ["suhii_all_sources", "resolve_source", "source_collection"],
    # decadal_product / trend_by_class_collection / decadal_band_order arrived
    # with the export-boundary restructure, so they are what distinguishes the
    # current trends.py from the one that failed in run 13.
    "trends": [
        "decadal_product", "trend_by_class_collection", "decadal_band_order",
        "selftest_annual_series", "annual_series", "require_annual_series",
        "fit_stack", "trend_image", "signum_array", "sens_slope_array",
        "mk_statistics_from_tau", "two_sided_p", "benjamini_hochberg",
        "fdr_significant_fraction", "mk_comparison", "suhii_trends",
        "decadal_means", "decadal_difference", "zonal_annual_series",
        "zonal_trend_table", "read_trend_raster", "apply_fdr_to_raster",
    ],
    "exports": [
        "export_name", "resolve_export_settings", "image_to_drive",
        "table_to_drive", "describe_tasks", "wait_for_tasks", "find_tasks",
    ],
    "landcover": ["stratified_stats_collection", "worldcover", "lcz_class_image",
                  "class_labels", "build_stratified_frame"],
    "viz": ["trend_vis_params", "build_trend_map_figure", "build_mk_comparison_figure",
            "build_trend_by_class_figure", "build_decadal_difference_figure"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  trends loaded from: {trends.__file__}\n"
        "Fix, in order:\n"
        "  1. MOST LIKELY: local changes are COMMITTED BUT NOT PUSHED, or not\n"
        "     committed at all. This notebook runs against the pushed repo, so\n"
        "     editing a file locally is not enough. Check that the HEAD line\n"
        "     printed by the clone cell is the revision you expect, then\n"
        "     git push and re-run from the CLONE cell.\n"
        "  2. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  3. If you uploaded this .ipynb by hand rather than opening it from\n"
        "     the repo, the notebook and src/ can be at DIFFERENT revisions -\n"
        "     that combination produces stale figures with no error at all."
    )
if "reducer_outputs" not in params["trends"]:
    raise RuntimeError(
        "config/params.yaml has no trends.reducer_outputs - your checkout "
        "predates the export-boundary restructure. Re-run the CLONE cell."
    )
print("PASS: all Phase 4 functions and params are present.")


# Run an interactive Earth Engine call; degrade to a note on a memory error.
# No interactive question about the trend graph is affordable (Colab runs
# 10-13). Everything that matters goes out through batch Export tasks, so an
# interactive call failing costs a convenience, never a result - and it must
# never abort the run.
def _try_ee(label, call, default=None):
    try:
        return call()
    except ee.EEException as error:
        if "memory" not in str(error).lower():
            raise
        print(f"SKIPPED ({label}): exceeded the interactive memory limit.")
        print("  Not a failure - the batch exports carry the real products.")
        return default


# Cheap params lookups, defined HERE so a skipped step downstream can never
# leave them undefined. The previous run died on exactly that cascade.
TREND_SOURCE = params["trends"]["pixel_sources"][0]
PIXEL_SOURCES = list(params["trends"]["pixel_sources"])
FIT_SCALE = params["trends"]["fit_scale_m"]
BANDS = params["trends"]["bands"]
FIRST_YEAR = params["time"]["start_year"]
LAST_YEAR = params["time"]["end_year"]

print()
print("Headline source :", TREND_SOURCE, "| fit scale:", FIT_SCALE, "m")
print("Pixel sources   :", PIXEL_SOURCES)
print("Reducer outputs :", params["trends"]["reducer_outputs"], "(measured, run 11)")

## Step 1 - geometries and the cheap water mask

| Decision | Why |
|---|---|
| `aoi.static_water_mask` | `aoi.water_mask` composites ~100 Landsat scenes internally and is re-instantiated for **every** image it masks. Over 26 years that is unaffordable. The measured cost of the substitution is **-0.074 degC** on the CMC 2025 mean - quote it, do not re-derive it. |
| `WORK_REGION` = Colombo District | `aoi.analysis_region` (Western Province + 25 km) is ~20x the pixels for nothing. |
| `FIT_SCALE` = 100 m | Adjacent 30 m LST pixels are near-duplicates, so fitting at 30 m multiplies the FDR test count ~11x without adding independent information - and drags the BH threshold down for every genuine pixel. |

In [ ]:
# COLAB: RUN THIS CELL
district_fc = aoi.colombo_district(params)
WORK_REGION = district_fc.geometry()
CMC = aoi.cmc_boundary(params)
STATIC_WATER = aoi.static_water_mask(params, region=WORK_REGION)
_box = CMC.centroid(maxError=1).buffer(1000).bounds()   # the probe box

print("District area (km2):", round(aoi.area_km2(WORK_REGION).getInfo(), 2))
print("CMC area (km2)     :", round(aoi.area_km2(CMC).getInfo(), 2))
print()
print("REMINDER: any CMC area must be quoted WITH its reduction scale - the")
print("land-only area is 40.18 km2 at 30 m and 37.70 km2 at 300 m.")

In [ ]:
# COLAB: RUN THIS CELL
# The name guard cannot see an ADDITIVE change. Phase 4 added a `series_basis`
# property to composites.annual_composites without renaming anything, so prove
# the SETTER is there. One cheap round trip on one year over the probe box.
_probe = composites.annual_composites(
    uhi_metrics.source_collection(TREND_SOURCE, params, region=_box),
    params, with_percentile=False, start_year=LAST_YEAR, end_year=LAST_YEAR,
).first()

_basis_prop = params["composites"]["series_basis_property"]
_basis = _probe.get(_basis_prop).getInfo()
print(f"{_basis_prop} = {_basis!r}")
if _basis != params["trends"]["series_basis"]:
    raise RuntimeError(
        f"composites.annual_composites did not set {_basis_prop} (got {_basis!r}). "
        "Your src/ predates Phase 4 - re-run the CLONE cell, then the purge cell."
    )
print("PASS: the Phase 4 series_basis marker is being set.")

## Step 2 - PROBE the reducers

Three unknowns, settled on a **tiny toy stack** rather than assumed. All three
have now been measured and are recorded in `config/params.yaml`; these cells
remain as the reproducible record, and as the place to re-measure if Earth
Engine ever changes.

1. **Band names.** `sensSlope` and `kendallsCorrelation` name their outputs
   differently depending on whether the reducer reports one input or several.
   Measured: with `numInputs=2` both emit **bare** names.
2. **Sen's input order.** `sensSlope` takes **x then y**. Reversed it returns
   the **reciprocal** slope - not an error. A known slope of 2.0 settles it.
3. **The reducer's own p-value.** Measured: it comes back `None`. Our
   `mk_p_two_sided` is derived from Z and is unaffected.

In [ ]:
# COLAB: RUN THIS CELL
# 2a - band names, on a 6-year stack over the ~2 km probe box.
_probe_series = trends.annual_series(
    TREND_SOURCE, params, region=_box, start_year=LAST_YEAR - 5, end_year=LAST_YEAR
)
_probe_stack = trends.fit_stack(
    _probe_series, params, start_year=LAST_YEAR - 5, validate=False
)

print("fit stack bands        :", _probe_stack.first().bandNames().getInfo())
print("sensSlope outputs      :",
      _probe_stack.reduce(ee.Reducer.sensSlope()).bandNames().getInfo())
print("kendallsCorrelation(2) :",
      _probe_stack.reduce(ee.Reducer.kendallsCorrelation(2)).bandNames().getInfo())
print("kendallsCorrelation(1) :",
      _probe_stack.select([trends.FIT_Y_BAND])
      .reduce(ee.Reducer.kendallsCorrelation(1)).bandNames().getInfo())
print()
print("Expected (measured run 11):", params["trends"]["reducer_outputs"])
print("If these differ, edit trends.reducer_outputs, commit, push, re-run from")
print("the CLONE cell - the product path selects by these names with no getInfo.")

In [ ]:
# COLAB: RUN THIS CELL
# 2b - Sen's input ORDER, against a collection whose slope is exactly 2.0.
# A reversed order returns the RECIPROCAL (0.5), not an error.
_known = ee.ImageCollection([
    ee.Image.cat([
        ee.Image.constant(x).toDouble().rename(trends.FIT_X_BAND),
        ee.Image.constant(10.0 + 2.0 * x).toDouble().rename(trends.FIT_Y_BAND),
    ])
    for x in (0, 1, 2, 3, 4)
])
_result = _known.reduce(ee.Reducer.sensSlope()).reduceRegion(
    reducer=ee.Reducer.first(), geometry=_box, scale=1000, maxPixels=1e9
).getInfo()
print("known slope 2.0 ->", _result)

_slope_value = next((v for k, v in _result.items() if "slope" in k.lower()), None)
if _slope_value is not None and abs(_slope_value - 2.0) < 1e-6:
    print("PASS: sensSlope takes x then y, as trends.sen_input_order assumes.")
elif _slope_value is not None and abs(_slope_value - 0.5) < 1e-6:
    raise RuntimeError(
        "FAIL: sensSlope returned the RECIPROCAL slope (0.5 for a true 2.0), so "
        "its inputs are y then x. Swap trends.sen_input_order in "
        "config/params.yaml to ['y', 'x'], commit, push, re-run from the CLONE cell."
    )
else:
    raise RuntimeError(f"FAIL: unexpected sensSlope output {_result}")

In [ ]:
# COLAB: RUN THIS CELL
# 2c - tau and the p-value, through the SAME code path the pipeline uses: an
# ImageCollection of two-band [x, y] images. Do NOT use ee.List.reduce here - a
# multi-input reducer over an ee.List wants a list of PAIRS, and two parallel
# arrays instead silently reduce 2 "samples" and return tau = 1.
import pymannkendall as pmk
from scipy import stats

_values = [26.1, 26.4, 26.3, 26.9, 27.0, 27.4, 27.2, 27.9, 28.1, 28.4, 28.3, 28.9]
_x = list(range(len(_values)))

_toy = ee.ImageCollection([
    ee.Image.cat([
        ee.Image.constant(float(_xi)).toDouble().rename(trends.FIT_X_BAND),
        ee.Image.constant(float(_yi)).toDouble().rename(trends.FIT_Y_BAND),
    ])
    for _xi, _yi in zip(_x, _values)
])
_ee_stats = _toy.reduce(ee.Reducer.kendallsCorrelation(2)).reduceRegion(
    reducer=ee.Reducer.first(), geometry=_box, scale=1000, maxPixels=1e9
).getInfo()
print("ee raw:", _ee_stats)


# Earth Engine returns NaN through JSON as the STRING 'NaN', not a float, so a
# bare `if value:` is truthy and the next arithmetic raises a TypeError.
def _as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


_ee_tau = _as_float(next((v for k, v in _ee_stats.items() if "tau" in k.lower()), None))
_ee_p = _as_float(
    next((v for k, v in _ee_stats.items() if "p" in k.lower() and "tau" not in k.lower()), None)
)

_sci = stats.kendalltau(_x, _values)
_pmk = pmk.original_test(_values)
_ours = trends.mk_statistics_from_tau(float(_sci.statistic), len(_values))

print()
print(f"  tau   scipy {float(_sci.statistic):.6f} | pmk {float(_pmk.Tau):.6f} | ee {_ee_tau:.6f}")
if not (np.isfinite(_ee_tau) and abs(_ee_tau - float(_sci.statistic)) < 1e-4):
    raise RuntimeError(
        f"FAIL: ee tau {_ee_tau} does not match scipy {float(_sci.statistic):.6f}. "
        "If ee tau is exactly 1 or -1 the inputs were not paired correctly."
    )
print("  PASS: ee tau matches scipy.")

# NOTE the two local p-values differ LEGITIMATELY: scipy's kendalltau uses an
# EXACT method at n=12, while pymannkendall - and our two_sided_p - use the
# normal approximation with a continuity correction. Ours must match pmk.
print()
print(f"  p  TWO-sided, normal approx (pmk) : {float(_pmk.p):.8f}")
print(f"  p  TWO-sided, ours from Z         : {float(_ours['p']):.8f}")
print(f"  p  ONE-sided would be             : {float(_pmk.p) / 2:.8f}")
print(f"  p  scipy EXACT (different method) : {float(_sci.pvalue):.8f}")
print(f"  p  ee reducer                     : {_ee_p}")
if abs(float(_ours["p"]) - float(_pmk.p)) > 1e-6:
    raise RuntimeError("FAIL: our two-sided p does not match pymannkendall.")
print()
print("  PASS: our two-sided p matches pymannkendall's normal approximation.")
if not np.isfinite(_ee_p):
    print("  ee p-value is NaN/None, as measured in run 11. NOT A PROBLEM:")
    print("  mk_p_ee is exported for comparison only and never reaches the FDR")
    print("  correction. Expect that band to be entirely masked.")

## Step 3 - the structural guard

The brief requires that running Mann-Kendall on the raw sub-annual stack be
**structurally impossible**. Two layers:

* **Layer 1** - `trends.trend_image()` takes a source *key*, not a collection,
  and builds its own annual series. There is no parameter through which a scene
  stack can reach the reducers.
* **Layer 2** - `trends.require_annual_series()` refuses anything that is not
  one image per year carrying the `series_basis` marker.

**We validate the CONSTRUCTOR, not the production series.** Interrogating the
26-year district series proved impossible across four attempts - reading even
two of its images' property *names* exceeded the memory limit. So the self-test
builds 3 years over the 2 km box through the **same** `annual_series()` call the
production path uses. A constructor that emits `series_basis`, omits `month`,
and yields one image per calendar year for 3 years over a small box does the
same for 26 over a district: the difference is data volume, not structure.

In [ ]:
# COLAB: RUN THIS CELL
# 3a - a SCENE collection must be rejected.
_scenes = uhi_metrics.source_collection(TREND_SOURCE, params, region=_box)
try:
    trends.require_annual_series(_scenes.limit(40), params)
except ValueError as error:
    print("PASS: scene collection rejected.")
    print("  ->", str(error).split(".")[0])
else:
    raise RuntimeError(
        "FAIL: require_annual_series ACCEPTED a scene collection. The structural "
        "guard is broken - do not run any trend product until it is fixed."
    )

print()
# 3b - the CONSTRUCTOR must produce a valid annual series.
_selftest = trends.selftest_annual_series(TREND_SOURCE, params, _box, years=3)
print("PASS: annual_series() builds a valid annual composite series.")
print("  ", _selftest)

## Step 4 - build the trend products (no inspection)

`trend_image()` builds with **zero** round trips: the reducer output band names
come from `trends.reducer_outputs`, measured in Step 2a rather than read back at
runtime.

**Nothing here inspects the result.** Band names, value ranges, percentiles and
maps all happen in Part 2, against the downloaded raster - because they can, and
because interactively they cannot.

In [ ]:
# COLAB: RUN THIS CELL
_t0 = time.time()
TREND_IMAGES = {}
for _key in PIXEL_SOURCES:
    TREND_IMAGES[_key] = trends.trend_image(
        _key, params, region=WORK_REGION
    ).clip(WORK_REGION)
    print("built trend image:", _key)

# The decadal product runs on the POOLED landsat_dry series ON PURPOSE. It is no
# longer a climate product - it is the SENSOR-STEP DIAGNOSTIC that revealed the
# discontinuity, and its 2000s/2010s/2020s windows only mean anything on the full
# 26-year record. Do not point it at landsat_oli_dry: that series starts in 2013,
# so two of the three windows would be empty.
DECADAL_SOURCE = "landsat_dry"
DECADAL = trends.decadal_product(
    DECADAL_SOURCE, params, region=WORK_REGION
).clip(WORK_REGION)
print("built decadal DIAGNOSTIC on the pooled series:", DECADAL_SOURCE)
print()
print("Decadal band order (this is what Part 2 reads the GeoTIFF back with):")
for _name in trends.decadal_band_order(params):
    print("   ", _name)
print()
print(f"({time.time() - _t0:.0f} s - all graph construction, no evaluation)")

## Step 5 - submit every export

Six batch tasks: three trend rasters, one decadal raster, and two by-class
tables. The tables run their grouped reducer **inside** the task, which is why
they work at all - the same reduction fails interactively.

`exports.image_to_drive` selects each image into an explicit band order before
exporting. Band identity in a GeoTIFF is **positional**, so that is what keeps
the writer and Part 2's reader describing the same file.

In [ ]:
# COLAB: RUN THIS CELL
TASKS = []

for _key, _img in TREND_IMAGES.items():
    _task = exports.image_to_drive(
        _img, product="lst_trend", aoi="district", params=params,
        region=WORK_REGION.bounds(),
        band_order=params["trends"]["export_band_order"],
        scale_m=FIT_SCALE, suffix=_key,
    )
    TASKS.append(_task)
    print("submitted:", _task.status().get("description"))

_decadal_task = exports.image_to_drive(
    DECADAL, product="lst_decadal", aoi="district", params=params,
    region=WORK_REGION.bounds(),
    band_order=trends.decadal_band_order(params),
    scale_m=FIT_SCALE, suffix=DECADAL_SOURCE,
)
TASKS.append(_decadal_task)
print("submitted:", _decadal_task.status().get("description"))

In [ ]:
# COLAB: RUN THIS CELL
# The by-class tables. The grouped reduction runs INSIDE the batch task, which
# is the only reason it is possible - interactively it exceeds the memory limit.
for _scheme in params["trends"]["stratify"]["products"]:
    _fc = trends.trend_by_class_collection(
        TREND_IMAGES[TREND_SOURCE], params, WORK_REGION, _scheme
    )
    _task = exports.table_to_drive(
        _fc, product="lst_trend_by", aoi="district", params=params,
        file_format="CSV",
        selectors=["class", "mean", "stdDev", "count"],
        res_m=FIT_SCALE, suffix=f"{_scheme}_{TREND_SOURCE}",
    )
    TASKS.append(_task)
    print("submitted:", _task.status().get("description"))

print()
print(f"{len(TASKS)} task(s) queued to Drive folder "
      f"'{params['exports']['drive_folder']}'.")
print("Re-run the NEXT cell until every state reads COMPLETED.")

In [ ]:
# COLAB: RUN THIS CELL  (re-runnable - poll until every state reads COMPLETED)
exports.describe_tasks(TASKS)

## Step 6 - cross-sensor continuity (run this; it decides whether to trust Landsat)

CLAUDE.md accepts that Collection 2 is inter-calibrated across TM/ETM+/OLI and
then adds: *"Still verify empirically on overlapping years."* **That check has
now been run, and it FAILED.**

| pair | offset over the CMC dry season | overlap yrs | t | verdict |
|---|---|---|---|---|
| L5 - L7 | **+1.78 degC** | 8 | +2.7 | **material** |
| L7 - L8 | **-2.48 degC** (L8 hotter) | 10 | -3.6 | **material** |
| L8 - L9 | -0.40 degC | 4 | -0.7 | negligible |

Those steps are **2.4x and 3.4x the entire 26-year trend signal** (+0.73 degC).
The L7->L8 step alone predicts a 2010s-minus-2000s difference of
0.8 x 2.48 = **+1.98 degC** against the **+1.70 degC** actually observed. The
pooled Landsat Mann-Kendall was measuring the changeover, not the climate.

**`trends.pixel_sources` now uses `landsat_oli_dry`** - Landsat 8 + 9 only,
2013-2025, one sensor family, no step. L8-L9 is negligible so they pool safely.
The cell below re-measures the offsets each run, because this is a property of
the data over your AOI, not a constant.

**How to read the numbers.** A non-zero offset is not automatically a
calibration error: two sensors observe on different dates in the same dry
season, so part of it is weather. The `sd_offset` here is 1.85-2.18 degC,
comparable to the offsets themselves, and the overlaps are only 8-10 years. What
makes the result actionable is that both material offsets are (a) significant,
(b) in the directions that reconcile with the observed decadal zigzag, and
(c) several times the trend they would contaminate. Any one of those alone would
be weak; together they are decisive.

**What still uses the pooled `landsat_dry` series, legitimately:**

* **SUHII** (Phase 3) - a within-year urban-minus-rural difference, so a
  common-mode sensor step cancels. Unaffected.
* **The decadal diagnostic** in Step 10 - it only means anything over the full
  26-year record, and it is now labelled as the thing that exposed the step
  rather than as a climate product.

In [ ]:
# COLAB: RUN THIS CELL
# Reduces SCENES in 4-year batches (the Phase 3 pattern), not the trend graph,
# so this is affordable interactively. Over the CMC, which is compact.
_t0 = time.time()
SENSOR_SERIES = _try_ee(
    "per-sensor annual means",
    lambda: trends.sensor_annual_means(
        params, CMC, water=STATIC_WATER, progress=True
    ),
)

if SENSOR_SERIES is None or SENSOR_SERIES.empty:
    print("Cross-sensor check unavailable this run.")
    SENSOR_OFFSETS = None
else:
    print(f"{len(SENSOR_SERIES)} rows in {time.time() - _t0:.0f} s")
    SENSOR_SERIES.to_csv("data/outputs/sensor_annual_means_cmc.csv", index=False)
    SENSOR_OFFSETS = trends.build_sensor_offset_summary(SENSOR_SERIES, params)
    SENSOR_OFFSETS.to_csv("data/outputs/sensor_offsets_cmc.csv", index=False)
    print("Wrote data/outputs/sensor_annual_means_cmc.csv")
    print("Wrote data/outputs/sensor_offsets_cmc.csv")
    display(SENSOR_OFFSETS)

In [ ]:
# COLAB: RUN THIS CELL
if SENSOR_OFFSETS is None or SENSOR_OFFSETS.empty:
    print("No offsets to interpret.")
else:
    _material = SENSOR_OFFSETS[
        SENSOR_OFFSETS["verdict"] == trends.SENSOR_OFFSET_MATERIAL
    ]
    print("=" * 72)
    print("VERDICT ON THE LANDSAT TREND PRODUCT")
    print("=" * 72)
    if _material.empty:
        print("No sensor pair shows a material offset over its overlap years.")
        print()
        print("C2 inter-calibration HOLDS in this AOI, so the zigzag is not a")
        print("sensor step and the zero significant pixels must be explained")
        print("some other way - most likely genuine interannual variability")
        print("swamping a ~0.03 degC/yr signal in a sparse, cloud-limited")
        print("dry-season series. Report the zero as a real (if weak) result.")
    else:
        print(f"{len(_material)} sensor pair(s) show a MATERIAL offset:")
        for _row in _material.itertuples():
            print(f"  {_row.sensor_a} vs {_row.sensor_b}: "
                  f"{_row.mean_offset:+.3f} degC over {_row.n_overlap_years} "
                  f"overlap years (t = {_row.t_statistic:.1f})")
        print()
        print("The Landsat 2000-2025 trend crosses these changeovers, so its")
        print("Mann-Kendall result measures the STEP as well as the climate.")
        print("Do NOT report '0% of Colombo shows significant warming'.")
        print()
        print("Report instead: single-sensor MODIS Terra night as the trend")
        print("evidence, the Landsat product as intra-urban PATTERN only, and")
        print("this table as the reason.")

## Step 6b - is the trend an artefact of GROWING OBSERVATION COUNTS?

Run 17 exposed a second problem, and it is not the sensor step. On the clean
single-sensor `landsat_oli_dry` series (L8+L9, 2013-2025) the median Sen's slope
is **-0.176 degC/yr** - about **-2.3 degC over 13 years**. That is no more
credible as cooling than the pooled series was as "no trend".

**The leading hypothesis is a sampling artefact.** Landsat 9 launched in late
2021, roughly doubling dry-season scene availability from 2022. An annual
composite is a MEDIAN over whatever clear-sky days existed:

* few scenes -> the median is pinned to a handful of clear days, and in a
  tropical dry season clear days are the HOT ones;
* many scenes -> the median regresses toward a more representative, cooler value.

So a growing constellation can manufacture apparent cooling with no change in
climate whatsoever. This cell tests that directly: if the annual mean LST tracks
`obs_count` downward, the trend is a sampling artefact, not a signal.

**If the correlation is strong and negative, the Landsat trend magnitude is not
reportable at any window** - and the deliverable falls back to the CONTRASTS
(Step 11), which are common-mode-cancelling and have already proved stable.

In [ ]:
# COLAB: RUN THIS CELL
# Reuses the Phase 3 batched zonal helper twice - once on the LST band, once on
# obs_count - over the CMC. Cheap, and it reduces composites, not the trend graph.
_obs_band = params["composites"]["obs_count_band"]
_oli = uhi_metrics.resolve_source(TREND_SOURCE, params)
_first, _last = trends.resolve_source_years(_oli, params)

_scenes_oli = uhi_metrics.source_collection(TREND_SOURCE, params, region=CMC)
_kw = dict(
    months=uhi_metrics.source_months(_oli, params),
    scale_m=FIT_SCALE, start_year=_first, end_year=_last,
    with_percentile=False, mask=STATIC_WATER,
)

OBS_CHECK = _try_ee(
    "obs_count vs LST",
    lambda: composites.zonal_annual_means_by_year(
        _scenes_oli, CMC, params, **_kw
    ).merge(
        composites.zonal_annual_means_by_year(
            _scenes_oli, CMC, params, band=_obs_band, **_kw
        ).rename(columns={"mean": "obs_count_mean"}),
        on=params["composites"]["year_property"], suffixes=("", "_obs"),
    ),
)

if OBS_CHECK is not None and not OBS_CHECK.empty:
    OBS_CHECK.to_csv("data/outputs/oli_obs_count_check_cmc.csv", index=False)
    print("Wrote data/outputs/oli_obs_count_check_cmc.csv")
    display(OBS_CHECK[[params["composites"]["year_property"], "mean",
                       "obs_count_mean", "valid_pixels"]])

In [ ]:
# COLAB: RUN THIS CELL
if OBS_CHECK is None or OBS_CHECK.empty:
    print("No obs_count series - hypothesis untested this run.")
else:
    _yr = params["composites"]["year_property"]
    _ok = OBS_CHECK.dropna(subset=["mean", "obs_count_mean"])
    _lst = _ok["mean"].to_numpy(dtype="float64")
    _obs = _ok["obs_count_mean"].to_numpy(dtype="float64")
    _years = _ok[_yr].to_numpy(dtype="float64")

    _r = float(np.corrcoef(_obs, _lst)[0, 1])
    _obs_slope, _ = trends.sens_slope_array(_years, _obs)
    _lst_slope, _ = trends.sens_slope_array(_years, _lst)

    print("=" * 72)
    print("SAMPLING-ARTEFACT TEST, CMC, ", int(_years.min()), "-", int(_years.max()))
    print("=" * 72)
    print(f"  obs_count trend : {_obs_slope:+.3f} scenes/yr")
    print(f"  LST trend       : {_lst_slope:+.4f} degC/yr")
    print(f"  corr(obs, LST)  : {_r:+.3f}  over {len(_ok)} years")
    print()
    if _r < -0.5 and _obs_slope > 0:
        print("  VERDICT: observation counts are RISING and LST falls with them.")
        print("  The apparent cooling is consistent with a SAMPLING ARTEFACT -")
        print("  a median over more clear-sky days regresses off the hot tail.")
        print("  DO NOT report the Landsat trend MAGNITUDE. Fall back to the")
        print("  class CONTRASTS in Step 11, which cancel a common-mode effect.")
    elif abs(_r) < 0.3:
        print("  VERDICT: LST and observation count are essentially uncorrelated,")
        print("  so the sampling hypothesis does NOT explain the decline. The")
        print("  cause is elsewhere - investigate before reporting any magnitude.")
    else:
        print("  VERDICT: partial association. Treat the magnitude as unreliable")
        print("  and report contrasts; note the correlation explicitly.")

## Step 7 - modified Mann-Kendall on the SUHII series (local)

This needs no trend graph at all - it works in pandas from Phase 3's exported
CSV - so it belongs in Part 1, where it gives the run a real result while the
exports queue.

Annual LST is positively autocorrelated, which inflates the true variance of S,
so the **uncorrected** Mann-Kendall p-value is anti-conservative. The Hamed &
Rao correction runs **beside** the plain test, and `var_inflation` (corrected
Var(S) / uncorrected) is the single number saying how much that mattered. If it
is near 1, the plain test was fine, and saying so is itself a result.

SUHII is decomposed into `urban_mean` and `rural_mean` too, because that answers
what the SUHII trend alone cannot: **did SUHII rise because the city warmed, or
because the countryside warmed less?**

In [ ]:
# COLAB: RUN THIS CELL
_suhii_csv = "data/outputs/suhii_2000_2025.csv"
if os.path.exists(_suhii_csv):
    SUHII = pd.read_csv(_suhii_csv)
    print("Loaded the Phase 3 SUHII table:", SUHII.shape)
else:
    print("Phase 3 SUHII CSV not found - rebuilding it (~42 round trips).")
    _pairs = uhi_metrics.mask_pairs(params, water=STATIC_WATER)
    SUHII = uhi_metrics.suhii_all_sources(params, pairs=_pairs, progress=True)
    os.makedirs("data/outputs", exist_ok=True)
    SUHII.to_csv(_suhii_csv, index=False)
    print("Wrote", _suhii_csv)

SUHII_TRENDS = trends.suhii_trends(SUHII, params)
os.makedirs("data/outputs", exist_ok=True)
SUHII_TRENDS.to_csv("data/outputs/suhii_trends_2000_2025.csv", index=False)
print("Wrote data/outputs/suhii_trends_2000_2025.csv")

SUHII_TRENDS[SUHII_TRENDS["series"] == "suhii"][
    ["label", "test", "n_years", "trend", "p", "slope", "var_inflation", "status"]
]

In [ ]:
# COLAB: RUN THIS CELL
os.makedirs("figures", exist_ok=True)
_fig = viz.plot_mk_comparison(
    SUHII_TRENDS[SUHII_TRENDS["series"] == "suhii"],
    "figures/mk_comparison_suhii_2000_2025.png",
    params,
    title="SUHII trend: effect of the autocorrelation correction",
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

---
# WAIT HERE

**Part 1 is done once every task in the status cell reads `COMPLETED`.**

Then get the files into `data/interim/`. Either:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/colombo_uhi_exports/*.tif data/interim/
!cp /content/drive/MyDrive/colombo_uhi_exports/*.csv data/interim/
```

or download them from Drive in a browser and upload into `data/interim/`.

A district raster export is not a 30-second wait, and a Colab cell blocked that
long risks the runtime disconnecting - which would lose the task handles as well
as the wait.

**To run Part 2 as a separate session**, re-run only these, then continue below:

1. the **clone** cell,
2. the **pip** cell (skip if the runtime is fresh from Part 1),
3. the **params + auth** cell,
4. **Step 0** - it carries every import and constant Part 2 needs,
5. **Step 1** geometries - only needed for Step 12's per-GN reduction.

Steps 2-7 do not need re-running: their outputs are already recorded, and Part 2
reads from `data/interim/`, not from Earth Engine.

---
# PART 2 - resume and analyse

Everything below is **local numpy and pandas**. No Earth Engine, except the one
clearly-marked per-GN reduction at the end, which works on annual composites in
4-year batches (the Phase 3 pattern that succeeded) rather than on the trend
graph - and which goes through `_try_ee` anyway.

This is where Phase 4's deliverables actually appear.

In [ ]:
# COLAB: RUN THIS CELL
# Locate the downloaded products and fail with an actionable message if absent.
os.makedirs("data/interim", exist_ok=True)

TREND_TIF = {}
for _key in PIXEL_SOURCES:
    _name = exports.export_name(
        "lst_trend", "district", params, res_m=FIT_SCALE, suffix=_key
    )
    _path = f"data/interim/{_name}.tif"
    TREND_TIF[_key] = _path if os.path.exists(_path) else None

DECADAL_SOURCE = "landsat_dry"      # the pooled series; a DIAGNOSTIC, not a product
DECADAL_TIF = f"data/interim/" + exports.export_name(
    "lst_decadal", "district", params, res_m=FIT_SCALE, suffix=DECADAL_SOURCE
) + ".tif"

print("Found in data/interim/:")
for _f in sorted(glob.glob("data/interim/*")):
    print("  ", _f)
print()
for _key, _path in TREND_TIF.items():
    print(f"  trend {_key:14s}: {'OK' if _path else 'MISSING'}")
print(f"  decadal            : {'OK' if os.path.exists(DECADAL_TIF) else 'MISSING'}")

if TREND_TIF[TREND_SOURCE] is None:
    raise FileNotFoundError(
        f"{TREND_TIF[TREND_SOURCE] or 'the headline trend raster'} is missing.\n"
        "1. Check every Part 1 task reads COMPLETED.\n"
        f"2. Copy the .tif and .csv files from Drive folder "
        f"'{params['exports']['drive_folder']}' into data/interim/ - see the\n"
        "   WAIT HERE cell above for the exact commands."
    )

## Step 8 - the FDR-corrected trend, from the exported raster

This is the authoritative product. Benjamini-Hochberg cannot be done
server-side - it needs every p-value at once, sorted - so it was always going to
happen here.

**Two denominators are reported and both matter.** Untested pixels (cloud
starved, water masked, or below the minimum-year floor) are neither significant
nor non-significant, so `fraction_of_tested` alone overstates coverage and
`fraction_of_total` alone understates the trend.

**Benjamini-Yekutieli is reported beside BH.** BH controls the FDR under
independence or positive regression dependency; a 100 m LST raster is strongly
spatially autocorrelated, so the *realised* false-discovery proportion varies far
more than its controlled expectation. BY is valid under arbitrary dependence and
is the honest upper bound. The pair is the sensitivity, exactly as Phase 3
reports the two rural definitions.

In [ ]:
# COLAB: RUN THIS CELL
_arrays, _profile = trends.read_trend_raster(TREND_TIF[TREND_SOURCE], params)
_slope = _arrays[BANDS["sen_slope"]]

print("Sen's slope percentiles over the exported raster (degC/yr):")
for _q in (1, 5, 25, 50, 75, 95, 99):
    print(f"  p{_q:<3d} {np.nanpercentile(_slope, _q): .4f}")

_vis = viz.trend_vis_params(params)
_saturated = np.nanmean(
    (_slope < _vis["min"]) | (_slope > _vis["max"])
) * 100
print()
print(f"Palette stretch: {_vis['min']} .. {_vis['max']} degC/yr")
print(f"Pixels outside it: {_saturated:.2f}%")
if _saturated > 5:
    print("  -> more than ~5% saturate. Widen trends.slope_vis: a saturated map")
    print("     understates the extremes it exists to show.")

In [ ]:
# COLAB: RUN THIS CELL
FDR = {}
for _key, _path in TREND_TIF.items():
    if _path is None:
        print(f"skipping {_key}: raster not downloaded")
        continue
    _rows = []
    for _method in params["trends"]["fdr"]["sensitivity_methods"]:
        _out = f"data/interim/{os.path.basename(_path)[:-4]}_fdr_{_method}.tif"
        _rows.append(trends.apply_fdr_to_raster(_path, _out, params, method=_method))
    FDR[_key] = pd.DataFrame(_rows)
    print(f"=== {_key} ===")
    display(FDR[_key][[
        "method", "n_total", "n_tested", "n_significant",
        "fraction_of_tested", "fraction_of_total",
        "n_warming", "n_cooling", "area_km2_significant",
    ]])

In [ ]:
# COLAB: RUN THIS CELL
_headline = FDR[TREND_SOURCE]
_bh = _headline[_headline["method"] == params["trends"]["fdr"]["method"]].iloc[0]
_by = _headline[_headline["method"] == "benjamini_yekutieli"].iloc[0]

print("=" * 72)
print("HEADLINE NUMBERS FOR THE REPORT -", TREND_SOURCE, f"at {FIT_SCALE} m")
print("=" * 72)
print(f"  tested            : {_bh['n_tested']:,} px "
      f"({_bh['area_km2_tested']:.1f} km2)")
print(f"  significant (BH)  : {_bh['n_significant']:,} px "
      f"({_bh['fraction_of_tested']:.1%} of tested, "
      f"{_bh['fraction_of_total']:.1%} of total)")
print(f"  significant (BY)  : {_by['n_significant']:,} px "
      f"({_by['fraction_of_tested']:.1%} of tested)")
print(f"  warming / cooling : {_bh['area_km2_warming']:.1f} / "
      f"{_bh['area_km2_cooling']:.1f} km2")
print()
print("Quote BH and BY TOGETHER, and always with the TESTED area. Untested")
print("pixels are neither significant nor non-significant.")

In [ ]:
# COLAB: RUN THIS CELL
for _key, _path in TREND_TIF.items():
    if _path is None:
        continue
    _fdr_tif = (f"data/interim/{os.path.basename(_path)[:-4]}"
                f"_fdr_{params['trends']['fdr']['method']}.tif")
    _a, _ = trends.read_trend_raster(
        _fdr_tif, params,
        band_order=["sen_slope", "sen_slope_fdr", "p_two_sided",
                    "p_adjusted", "significant", "n_years"],
    )
    _fig = viz.plot_trend_map(
        _a, f"figures/trend_fdr_{_key}_2000_2025.png", params,
        title=f"Sen's slope, {_key}, 2000-2025 (FDR-corrected, {FIT_SCALE} m)",
    )
    print("Wrote", _fig)
    display(Image(filename=str(_fig)))

## Step 9 - MODIS night, and what it is worth

Landsat sees a single ~10:30 overpass, so **night-time UHI can only come from
MODIS** (caveat 4). Two caveats travel with those maps:

* MODIS is a **coarse-unit** statistic - roughly 40 MODIS pixels cover the whole
  CMC, and they are edge-contaminated.
* Day and night are **not an equal-confidence pair**: night LST carries up to 3 K
  accepted uncertainty against 1 K for day.

MODIS *daytime* is deliberately absent from `trends.pixel_sources`: Terra's
orbital drift after ~2020 moves the overpass time and contaminates end-of-series
daytime trends. Night is unaffected by that drift. The maps were drawn in the
loop above; the numbers are in the FDR table.

## Step 10 - the decadal SENSOR-STEP DIAGNOSTIC

**This is not a climate product.** It runs on the POOLED `landsat_dry` series -
the one Step 6 showed is contaminated by cross-sensor steps - because its
2000s/2010s/2020s windows only mean anything over the full 26-year record. Its
job now is to show the discontinuity, which is what it did: **+1.70 degC then
-1.23 degC**, a zigzag whose first leg the L7->L8 offset predicts to within
0.3 degC.

Read it as evidence about the SENSORS. Do not quote either number as warming.

**The windows are 11 / 10 / 5 years - unequal by construction**, because the
study period ends in 2025. The 2021-2025 mean rests on roughly half the sample
of the others, so its standard error is about sqrt(2) larger and it dominates
any difference it appears in. That is why the figure draws the difference
**beside** its signal-to-noise panel.

**The headline warming number is the Sen's slope, not a decadal difference.** A
difference conflates trend with interannual variability (one hot year at either
end moves it) and with changing observation counts. These maps show *where* the
warming is concentrated.

In [ ]:
# COLAB: RUN THIS CELL
if not os.path.exists(DECADAL_TIF):
    print("Decadal raster not downloaded - skipping Step 10.")
else:
    _dec, _ = trends.read_trend_raster(
        DECADAL_TIF, params, band_order=trends.decadal_band_order(params)
    )
    _labels = [label for label, _, _ in trends.resolve_decades(None, params)]
    for _later, _earlier in zip(_labels[1:], _labels):
        _tag = f"{_later}_minus_{_earlier}"
        _fig = viz.plot_decadal_difference(
            _dec, f"figures/trend_decadal_{_tag}.png", params,
            difference_key=f"diff_{_tag}", se_key=f"diff_se_{_tag}",
            title=f"Mean LST {_later.replace('_', '-')} minus "
                  f"{_earlier.replace('_', '-')}",
        )
        _d = _dec[f"diff_{_tag}"]
        _z = _dec[f"diff_z_{_tag}"]
        _n = _dec[f"n_years_min_{_tag}"]
        print(f"{_tag}")
        print(f"  difference : median {np.nanmedian(_d):+.3f} degC "
              f"(p5 {np.nanpercentile(_d, 5):+.3f}, p95 {np.nanpercentile(_d, 95):+.3f})")
        # The difference alone is not interpretable with 11/10/5-year windows.
        # |z| > 2 is roughly "distinguishable from interannual noise".
        print(f"  diff / SE  : median {np.nanmedian(_z):+.2f} "
              f"| {np.nanmean(np.abs(_z) > 2) * 100:.1f}% of pixels |z| > 2")
        print(f"  min years  : median {np.nanmedian(_n):.0f} "
              f"(the SHORTER window limits every pixel here)")
        print("  wrote", _fig)
        display(Image(filename=str(_fig)))

    print()
    print("A decadal difference is NOT the warming rate - Sen's slope is. And if")
    print("the two differences have OPPOSITE signs, the series is not monotonic,")
    print("which is what Mann-Kendall tests. Read that against Step 6.")

## Step 11 - trend magnitude by land cover and by LCZ

**These are a present-day stratification of a historical trend.** ESA WorldCover
is 2021 and the LCZ map derives from 2018-2019 imagery, so both answer *"where
is the warming, by TODAY'S land cover?"* - not *"did land-cover change cause the
warming?"*. A pixel that was paddy in 2002 and is built now sits in the built
class for its whole history. Attribution to land-cover change is Phase 6.

In [ ]:
# COLAB: RUN THIS CELL
CLASS_TABLES = {}
for _scheme in params["trends"]["stratify"]["products"]:
    _name = exports.export_name(
        "lst_trend_by", "district", params, res_m=FIT_SCALE,
        suffix=f"{_scheme}_{TREND_SOURCE}",
    )
    _csv = f"data/interim/{_name}.csv"
    if not os.path.exists(_csv):
        print(f"skipping {_scheme}: {_csv} not downloaded")
        continue

    # The exported CSV holds the raw grouped-reducer output; build_stratified_frame
    # is the SAME shaping the interactive path uses, so both routes agree.
    _raw = pd.read_csv(_csv)
    _groups = _raw.rename(columns={"class": "class"}).to_dict("records")
    _table = landcover.build_stratified_frame(
        _groups, params, _scheme, ["mean", "stdDev"]
    )
    CLASS_TABLES[_scheme] = _table
    _out = f"data/outputs/lst_trend_by_{_scheme}_2000_2025.csv"
    _table.to_csv(_out, index=False)
    print("Wrote", _out)
    display(_table)

    _fig = viz.plot_trend_by_class(
        _table, f"figures/trend_by_{_scheme}_2000_2025.png", params
    )
    print("Wrote", _fig)
    display(Image(filename=str(_fig)))

In [ ]:
# COLAB: RUN THIS CELL
# CONTRASTS, not absolute slopes. A common-mode bias - a sensor step, a
# scale-dependent offset, an atmospheric-correction drift - shifts every class
# by the same amount and therefore CANCELS in a class-to-class ratio, while it
# survives untouched in an absolute degC/yr figure. Rank and ratio are the
# robust statements; quote those first and the absolute slope second.
for _scheme, _table in CLASS_TABLES.items():
    _usable = _table[~_table["below_pixel_floor"]].copy()
    if _usable.empty:
        continue
    _reference = _usable.loc[_usable["mean"].idxmin()]
    print(f"=== {_scheme}: warming RELATIVE to the coolest well-sampled class ===")
    print(f"  reference: {_reference['class_label']} "
          f"({_reference['mean']:+.4f} degC/yr, n={int(_reference['pixel_count']):,})")
    print()
    for _row in _usable.sort_values("mean", ascending=False).itertuples():
        _delta = _row.mean - _reference["mean"]
        print(f"  {_row.class_label:28s} {_row.mean:+.4f}  "
              f"(reference {_delta:+.4f} degC/yr, n={int(_row.pixel_count):,})")
    print()

## Step 12 - per-GN trends and the MAUP sensitivity

The one remaining server-side call. It reduces the **annual composites** in
4-year batches - the Phase 3 pattern that succeeded - not the trend graph, so it
should work. It goes through `_try_ee` regardless.

FDR is applied **across divisions** here, where `m` is about 557 rather than a
hundred thousand and the tests are far less mutually dependent. The pixel-wise
and per-GN significant fractions **will** differ, and that spread is the
aggregation-unit (MAUP) sensitivity CLAUDE.md requires, applied to significance
rather than to means.

In [ ]:
# COLAB: RUN THIS CELL
_t0 = time.time()
GN_SERIES = _try_ee(
    "per-GN annual series",
    lambda: trends.zonal_annual_series(
        TREND_SOURCE, params, level="gn", region=WORK_REGION, progress=True
    ),
)

if GN_SERIES is None or GN_SERIES.empty:
    print("No per-GN series - the MAUP comparison is unavailable this run.")
else:
    print(f"{len(GN_SERIES)} rows in {time.time() - _t0:.0f} s")
    GN_TRENDS = trends.zonal_trend_table(GN_SERIES, params)
    GN_SERIES.to_csv("data/outputs/lst_by_gn_annual_2000_2025.csv", index=False)
    GN_TRENDS.to_csv("data/outputs/lst_trend_by_gn_2000_2025.csv", index=False)
    print("Wrote data/outputs/lst_by_gn_annual_2000_2025.csv")
    print("Wrote data/outputs/lst_trend_by_gn_2000_2025.csv")

    _sig = int(GN_TRENDS["significant"].sum())
    print()
    print(f"GN divisions with an FDR-significant trend: {_sig} of {len(GN_TRENDS)} "
          f"({_sig / max(len(GN_TRENDS), 1):.1%})")
    if "_bh" in dir():
        print(f"PIXEL-wise fraction (BH, {TREND_SOURCE}): "
              f"{_bh['fraction_of_tested']:.1%} of tested")
        print()
        print("Those two differ by construction. Report them as a PAIR - the")
        print("spread is the aggregation-unit (MAUP) sensitivity applied to")
        print("significance.")
    else:
        print("(Run Step 8 for the pixel-wise fraction to compare against.)")
    display(GN_TRENDS.sort_values("slope", ascending=False).head(10))

## What to check before signing Phase 4 off

1. **Step 6 FIRST** - the cross-sensor verdict. `landsat_oli_dry` (L8+L9,
   2013-2025) is now the configured Landsat source precisely because this check
   failed on the pooled series. Confirm L8-L9 is still `negligible`; if it ever
   turns `material`, even the single-sensor-family product needs splitting.
2. **Step 2a** - the probe's band names still match `trends.reducer_outputs`.
3. **Step 2b** - `sensSlope` returned 2.0, not 0.5.
4. **Step 3** - the guard REJECTED the scene collection and the constructor
   self-test passed. If it accepted both, stop: the guard is broken.
5. **Step 5** - all six tasks reached `COMPLETED`.
6. **Step 8** - slope percentiles do not saturate the palette; BH and BY
   fractions are both recorded, with the tested area. A large BH/BY gap is the
   dependence sensitivity, not a discrepancy to resolve.
7. **Step 10** - do the two decadal differences have OPPOSITE signs? If so the
   series is not monotonic and the Mann-Kendall result must be read against
   Step 6 before it is quoted.
8. **Step 7** - `var_inflation`. If it is exactly 1.000 everywhere, Hamed-Rao
   found no significant autocorrelation lag - report that as a result, not as a
   correction that was applied.
9. **Step 12** - the pixel-wise and per-GN fractions, reported as a pair.

## What must NOT be claimed

* This is **land surface temperature**. Not air temperature, not "what residents
  feel". Surface UHI can be roughly 2x the canopy-air UHI.
* **NOTHING may be quoted from a Landsat trend fitted across 2013.** Step 6
  measured L5-L7 = +1.78 degC and L7-L8 = -2.48 degC over the CMC dry season -
  2.4x and 3.4x the entire 26-year trend signal. The Landsat trend claim is
  **2013-2025, Landsat 8+9 only**, and must be stated with that window.
* Because that window is 13 years, the Landsat trend is **lower-powered** than
  the MODIS one and its `n_tested` fraction is smaller. Report both.
* The Landsat slope is a **dry-season, ~10:30 overpass** rate. The MODIS night
  slope is a different quantity, reported beside it as a sensitivity - and it is
  weaker evidence (up to 3 K accepted uncertainty against 1 K for day).
* The significance map **inherits WRS-2 side-lap striping** from `obs_count`,
  one strip of which crosses the CMC. Report it; do not tune it away.
* Trend stratified by land cover is **not** attribution to land-cover change.
  Prefer the **contrast** ("built-up warms ~Nx faster than tree cover") to the
  absolute degC/yr: a common-mode bias cancels in the ratio and survives in the
  absolute.
* A **decadal difference is not the warming rate**, and the decadal product here
  is a sensor diagnostic, not a climate product.